# OpenStroke15 / E99: instance queries, chỉ GT

Chạy lần lượt từ trên xuống trên VM GPU mới. Ô 3 tự kiểm ZIP và checkpoint E69, báo rõ nếu thiếu.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
%pip install -q --force-reinstall --no-deps "git+https://github.com/Nguyenchitrai62/OpenStroke.git@main"
%pip install -q scipy shapely opencv-python pyyaml matplotlib

In [ ]:
from pathlib import Path
from zipfile import ZipFile
import subprocess

ZIP = Path("/content/drive/MyDrive/Dataset/OpenStroke9/openstroke_hvac7_data.zip")
MODEL = Path("/content/drive/MyDrive/Train/OpenStroke9/e69/best.pt")   # checkpoint E69; sửa nếu để chỗ khác

if not MODEL.is_file():
    found = sorted(Path("/content/drive/MyDrive").rglob("*.pt"))
    print("Không thấy MODEL. Các file .pt trên Drive:")
    for f in found[:200]:
        print("  ", f)
    raise FileNotFoundError(f"Sửa MODEL thành đường dẫn checkpoint E69 trong danh sách trên: {MODEL}")
assert ZIP.is_file(), f"Thiếu ZIP dữ liệu: {ZIP}"

ROOT = Path("/content/openstroke9_e83")
if not (ROOT / "obb_hvac7/data.yaml").is_file():
    with ZipFile(ZIP) as archive:
        archive.extractall(ROOT)
subprocess.run(["python", "-m", "openstroke.obb.gt_dataset",
                "--source", str(ROOT / "obb_hvac7"), "--out", "/content/openstroke15_gt"], check=True)
print("MODEL =", MODEL)

## Pilot 10 epoch (lịch LR của 50 epoch)

In [ ]:
!openstroke obb train   data=/content/openstroke15_gt/data.yaml   model={MODEL}   epochs=50   pilot_epochs=10   batch=32   nbs=32   lr0=0.0001   warmup_epochs=0.5   seed=0   imgsz=1024   tile=True   tile_scale=1   max_tokens=8192   max_iters=190   mosaic=0   scale=0.25   instance_queries=256   instance_free=128   instance_line_context=True   instance_f2=False   stroke_graph=False   patience=0   project=/content/drive/MyDrive/Train/OpenStroke15   name=e99_instances_gt

## Chỉ chạy sau khi xem log pilot: train tiếp tới 50 epoch

Tự tìm run e99_instances_gt chưa xong mới nhất (kể cả hậu tố 2, 3...).

In [ ]:
!openstroke obb train   resume=True   project=/content/drive/MyDrive/Train/OpenStroke15   name=e99_instances_gt   pilot_epochs=0